<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/openai_assistant_api_chatpdf_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI Assistants API로 ChatPDF 만들기

## Assistants API Reference : https://platform.openai.com/docs/api-reference/assistants/createAssistant

In [1]:
!pip install openai

In [2]:
!pip show openai

Name: openai
Version: 2.14.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: 


## OpenAI API Key 설정

In [3]:
from openai import OpenAI

# API 키 입력 및 클라이언트 생성
client = OpenAI(api_key="Input Your API Key")

# OpenAI에 File 업로드

## 한국의 저출산 관련 PDF : https://snuac.snu.ac.kr/2015_snuac/wp-content/uploads/2015/07/asiabrief_3-26.pdf

In [8]:
file = client.files.create(
  file=open("저출산.pdf", "rb"),
  purpose="assistants"
)
file

FileObject(id='file-NDnBGpTn8W5Vo1pQnBr8rS', bytes=615188, created_at=1768979038, filename='저출산.pdf', object='file', purpose='assistants', status='processed', expires_at=None, status_details=None)

In [9]:
#파일 ID
file.id

'file-NDnBGpTn8W5Vo1pQnBr8rS'

# Step 1: Create an Assistant

In [14]:
assistant = client.beta.assistants.create(
    name="ChatPDF",
    instructions="PDF 안의 내용을 참조해서 사용자의 질문에 답변해줘.", #PDF 기반으로 답변하게끔 설정
    model="gpt-4o-mini",
    tools=[{"type": "file_search"}],  # 구버전 retrieval -> 신버전 file_search 변경
    tool_resources={
        "file_search": {
            "vector_stores": [{
                "file_ids": [file.id]  # file_ids는 이제 tool_resources 안에 넣어야 합니다
            }]
        }
    }
)

# Step 2: Create a Thread

In [15]:
thread = client.beta.threads.create()
thread

/tmp/ipython-input-1159370957.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread = client.beta.threads.create()


Thread(id='thread_tDNbBxR6mzT6zdgkqAb1TcEA', created_at=1768979166, metadata={}, object='thread', tool_resources=ToolResources(code_interpreter=None, file_search=None))

# Step 3: Add a Message to a Thread

In [16]:
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="한국의 저출산의 원인이 무엇이야?"
)
message

/tmp/ipython-input-35944230.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  message = client.beta.threads.messages.create(


Message(id='msg_MHBv0eNfrSqq9hXowy3nd8Xt', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='한국의 저출산의 원인이 무엇이야?'), type='text')], created_at=1768979173, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_tDNbBxR6mzT6zdgkqAb1TcEA')

# Step 4: Run the Assistant

In [17]:
run = client.beta.threads.runs.create(
  thread_id=thread.id,
  assistant_id=assistant.id
)
run

/tmp/ipython-input-1909469769.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.create(


Run(id='run_J4fe3JUHCrp73sakgdMVl2Q7', assistant_id='asst_MAGnBUnrmnsu6ddbXoprmPAH', cancelled_at=None, completed_at=None, created_at=1768979180, expires_at=1768979780, failed_at=None, incomplete_details=None, instructions='PDF 안의 내용을 참조해서 사용자의 질문에 답변해줘.', last_error=None, max_completion_tokens=None, max_prompt_tokens=None, metadata={}, model='gpt-4o-mini', object='thread.run', parallel_tool_calls=True, required_action=None, response_format='auto', started_at=None, status='queued', thread_id='thread_tDNbBxR6mzT6zdgkqAb1TcEA', tool_choice='auto', tools=[FileSearchTool(type='file_search', file_search=FileSearch(max_num_results=None, ranking_options=FileSearchRankingOptions(score_threshold=0.0, ranker='default_2024_08_21', hybrid_search=None)))], truncation_strategy=TruncationStrategy(type='auto', last_messages=None), usage=None, temperature=1.0, top_p=1.0, tool_resources={}, reasoning_effort=None)

# Step 5: Check the Run status

In [18]:
run = client.beta.threads.runs.retrieve(
  thread_id=thread.id,
  run_id=run.id
)
run.status

/tmp/ipython-input-2535962091.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.retrieve(


'completed'

# Step 6: Display the Assistant's Response

In [19]:
#업로드한 PDF 파일을 기반으로 답변을 하였다
messages = client.beta.threads.messages.list(
  thread_id=thread.id
)
messages

/tmp/ipython-input-3204348557.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(


SyncCursorPage[Message](data=[Message(id='msg_YwBxUkiPz64vDv4Jn18QZwtn', assistant_id='asst_MAGnBUnrmnsu6ddbXoprmPAH', attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[FileCitationAnnotation(end_index=259, file_citation=FileCitation(file_id='file-NDnBGpTn8W5Vo1pQnBr8rS'), start_index=247, text='【4:0†source】', type='file_citation'), FileCitationAnnotation(end_index=421, file_citation=FileCitation(file_id='file-NDnBGpTn8W5Vo1pQnBr8rS'), start_index=409, text='【4:2†source】', type='file_citation'), FileCitationAnnotation(end_index=433, file_citation=FileCitation(file_id='file-NDnBGpTn8W5Vo1pQnBr8rS'), start_index=421, text='【4:6†source】', type='file_citation'), FileCitationAnnotation(end_index=570, file_citation=FileCitation(file_id='file-NDnBGpTn8W5Vo1pQnBr8rS'), start_index=558, text='【4:3†source】', type='file_citation'), FileCitationAnnotation(end_index=582, file_citation=FileCitation(file_id='file-NDnBGpTn8W5Vo1pQnBr8rS'), start_index=570, text='【4:0†

In [ ]:
len(messages.data)

2

In [20]:
for idx, message in enumerate(messages.data):
  print(f'-- {idx} ----------------------------------------------------------')
  print(message.content[0].text.value)

-- 0 ----------------------------------------------------------
한국의 저출산 현상은 여러 복잡한 요인들이 얽혀 있는 결과로 나타나고 있습니다. 주요 원인으로는 다음과 같은 것들이 있습니다:

1. **결혼관의 변화**: 한국에서는 여전히 전통적인 가치관이 존재하여 법률혼을 전제로 한 가족 형성이 중시됩니다. 결혼은 안정된 직장, 주거지, 학력 등의 조건을 충족해야 가능하다고 여겨지며, 이러한 조건을 만족시키는 것이 점점 더 어려워지고 있습니다. 그로 인해 결혼 자체를 기피하는 경향이 증가하고 있습니다【4:0†source】.

2. **경제적 요인**: 양육 비용의 부담이 크고, 불안정한 고용 상황, 그리고 높은 주거비용 등이 출산을 꺼리게 만드는 요인으로 작용하고 있습니다. 자녀 양육 비용은 의료비, 보육비, 교육비 등 다양한 형태로 발생하며, 이는 가계에 상당한 부담을 주고 있습니다【4:2†source】【4:6†source】.

3. **사회적 경쟁**: 한국 사회의 과도한 경쟁 역시 저출산의 원인으로 지적됩니다. 어린 시절부터의 무한 경쟁이 아이를 낳고 양육하는 것에 대한 부담으로 작용하여, 자녀 양육을 행복이 아닌 부담으로 인식하게 만듭니다【4:3†source】【4:0†source】.

4. **성평등 문제**: 여전히 존재하는 성차별과 여성의 노동 시장에서의 차별도 저출산에 기여하고 있습니다. 여성의 경제활동 참여가 증가했음에도 불구하고, 일과 가정 양립에 대한 사회적 지원이 부족하여 경력 단절 우려가 큰 상황입니다【4:2†source】【4:5†source】.

이와 같은 요인들이 서로 긴밀하게 연결되어 있으며, 이 문제를 해결하기 위해서는 종합적이고 장기적인 접근이 필요합니다. 저출산 대책은 단순히 출산 장려 정책을 넘어서, 경제적 안정과 일자리, 그리고 성평등을 모두 아우르는 방향으로 추진되어야 합니다【4:1†source】【4:4†source】.
-- 1 ------------------------

In [21]:
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="PDF 파일안에 소개된 한국의 저출산 관련 최신 관련 자료를 찾아줘"
)
message

/tmp/ipython-input-3753327004.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  message = client.beta.threads.messages.create(


Message(id='msg_laANXevQNJ0OugD9GyQ1jbz7', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='PDF 파일안에 소개된 한국의 저출산 관련 최신 관련 자료를 찾아줘'), type='text')], created_at=1768979270, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_tDNbBxR6mzT6zdgkqAb1TcEA')

In [22]:
run = client.beta.threads.runs.create(
  thread_id=thread.id,
  assistant_id=assistant.id
)
run

/tmp/ipython-input-1909469769.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.create(


Run(id='run_ZbmdVZo3wKslq3fQcUXpzb44', assistant_id='asst_MAGnBUnrmnsu6ddbXoprmPAH', cancelled_at=None, completed_at=None, created_at=1768979277, expires_at=1768979877, failed_at=None, incomplete_details=None, instructions='PDF 안의 내용을 참조해서 사용자의 질문에 답변해줘.', last_error=None, max_completion_tokens=None, max_prompt_tokens=None, metadata={}, model='gpt-4o-mini', object='thread.run', parallel_tool_calls=True, required_action=None, response_format='auto', started_at=None, status='queued', thread_id='thread_tDNbBxR6mzT6zdgkqAb1TcEA', tool_choice='auto', tools=[FileSearchTool(type='file_search', file_search=FileSearch(max_num_results=None, ranking_options=FileSearchRankingOptions(score_threshold=0.0, ranker='default_2024_08_21', hybrid_search=None)))], truncation_strategy=TruncationStrategy(type='auto', last_messages=None), usage=None, temperature=1.0, top_p=1.0, tool_resources={}, reasoning_effort=None)

In [23]:
run = client.beta.threads.runs.retrieve(
  thread_id=thread.id,
  run_id=run.id
)
run.status

/tmp/ipython-input-2535962091.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.retrieve(


'in_progress'

In [24]:
messages = client.beta.threads.messages.list(
  thread_id=thread.id
)
messages

/tmp/ipython-input-3204348557.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(


SyncCursorPage[Message](data=[Message(id='msg_QCbh8BUYV2RR2dbOOWAXTCYe', assistant_id='asst_MAGnBUnrmnsu6ddbXoprmPAH', attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[FileCitationAnnotation(end_index=386, file_citation=FileCitation(file_id='file-NDnBGpTn8W5Vo1pQnBr8rS'), start_index=374, text='【8:0†source】', type='file_citation')], value='한국의 저출산 관련 최신 자료로는 다음과 같은 문헌들이 소개됩니다:\n\n1. 이삼식 외 (2021). "저출산·고령사회의 효율적 대응을 위한 추진체계 구축방안." 대통령직속 저출산고령사회위원회, 한양대학교 고령사회연구원.\n2. 이삼식 외 (2020). "저출산에 따른 재정부담 분석 및 대응." 기획재정부, 한양대학교 고령사회연구원.\n3. 이삼식 (2020). "한국 인구정책 변천과 시대적 함의." 기획재정부, KDI 글로벌지식협력단지.\n4. 이삼식 외 (2018). "대한민국 중장기 인구정책 방향." 보건복지부, 한양대학교 고령사회연구원.\n5. 이태훈 (2019). "프랑스 출산율 상승에 긍정적인 영향을 미친 가족정책." 『국제노동브리프』 12월호, 한국노동연구원【8:0†source】.\n\n이 자료들은 저출산 문제를 다룬 연구 및 정책 제안들을 포함하고 있어, 관련 논의에 유용한 참고자료가 될 것입니다.'), type='text')], created_at=1768979282, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='assistant', run_id='run_ZbmdVZo

In [25]:
for idx, message in enumerate(messages.data):
  print(f'-- {idx} ----------------------------------------------------------')
  print(message.content[0].text.value)

-- 0 ----------------------------------------------------------
한국의 저출산 관련 최신 자료로는 다음과 같은 문헌들이 소개됩니다:

1. 이삼식 외 (2021). "저출산·고령사회의 효율적 대응을 위한 추진체계 구축방안." 대통령직속 저출산고령사회위원회, 한양대학교 고령사회연구원.
2. 이삼식 외 (2020). "저출산에 따른 재정부담 분석 및 대응." 기획재정부, 한양대학교 고령사회연구원.
3. 이삼식 (2020). "한국 인구정책 변천과 시대적 함의." 기획재정부, KDI 글로벌지식협력단지.
4. 이삼식 외 (2018). "대한민국 중장기 인구정책 방향." 보건복지부, 한양대학교 고령사회연구원.
5. 이태훈 (2019). "프랑스 출산율 상승에 긍정적인 영향을 미친 가족정책." 『국제노동브리프』 12월호, 한국노동연구원【8:0†source】.

이 자료들은 저출산 문제를 다룬 연구 및 정책 제안들을 포함하고 있어, 관련 논의에 유용한 참고자료가 될 것입니다.
-- 1 ----------------------------------------------------------
PDF 파일안에 소개된 한국의 저출산 관련 최신 관련 자료를 찾아줘
-- 2 ----------------------------------------------------------
한국의 저출산 현상은 여러 복잡한 요인들이 얽혀 있는 결과로 나타나고 있습니다. 주요 원인으로는 다음과 같은 것들이 있습니다:

1. **결혼관의 변화**: 한국에서는 여전히 전통적인 가치관이 존재하여 법률혼을 전제로 한 가족 형성이 중시됩니다. 결혼은 안정된 직장, 주거지, 학력 등의 조건을 충족해야 가능하다고 여겨지며, 이러한 조건을 만족시키는 것이 점점 더 어려워지고 있습니다. 그로 인해 결혼 자체를 기피하는 경향이 증가하고 있습니다【4:0†source】.

2. **경제적 요인**: 양육 비용의 부담이 크고, 불안정한 고용 상황, 그리고 높은 주거비용 